# Library Loan System & Data Processing Engine

## 1. Project Setup & Configuration

In [1]:
import csv
import datetime
from pathlib import Path
from collections import defaultdict, Counter

# Define directory paths relative to repo root
DATA_DIR = Path("data")
REPORTS_DIR = Path("reports")

# Ensure output directories exist
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Excel Epoch Constant
EXCEL_EPOCH_START = datetime.datetime(1899, 12, 30)

def excel_to_date_string(excel_date: int) -> str:
    """Converts an Excel integer epoch date to a YYYY-MM-DD formatted string."""
    try:
        py_date = EXCEL_EPOCH_START + datetime.timedelta(days=int(excel_date))
        return py_date.strftime('%Y-%m-%d')
    except (ValueError, TypeError):
        return "Invalid Date"

## 2. Object-Oriented Domain Models

In [2]:
class Book:
    """Represents an individual library book and its catalog metadata."""
    def __init__(self, book_id: int, title: str, author: str, genre: str):
        self.book_id = int(book_id)
        self.title = title.strip()
        self.author = author.strip()
        self.genre = genre.strip()

    def __repr__(self):
        return f"Book({self.book_id}, '{self.title}', '{self.genre}')"


class Bookshelf:
    """Collection manager for library catalog data."""
    def __init__(self):
        self.books = {}

    def load_catalog(self, filepath: Path):
        """Loads and parses the books catalog CSV file."""
        with open(filepath, 'r', encoding='utf-8-sig') as f:
            reader = csv.reader(f)
            for row in reader:
                if not row or len(row) < 4:
                    continue
                try:
                    book_id, title, author, genre = row[0], row[1], row[2], row[3]
                    self.books[int(book_id)] = Book(book_id, title, author, genre)
                except ValueError:
                    continue  # Skip header or malformed rows

    def get_book(self, book_id: int) -> Book:
        return self.books.get(book_id)


class LoanData:
    """Encapsulates a single cleaned loan transaction record."""
    def __init__(self, book_id: int, member_id: int, date_loaned: int, date_returned: int):
        self.book_id = int(book_id)
        self.member_id = int(member_id)
        self.date_loaned_raw = int(date_loaned)
        self.date_returned_raw = int(date_returned)

        # Computed Properties
        self.date_loaned_str = excel_to_date_string(self.date_loaned_raw)
        self.date_returned_str = excel_to_date_string(self.date_returned_raw)
        self.days_loaned = (self.date_returned_raw - self.date_loaned_raw) + 1
        self.days_late = max(0, self.days_loaned - 14)
        self.is_late = self.days_late > 0

## 3. Data Cleaning Pipeline & Preprocessing

In [3]:
def process_loan_records(filepath: Path, max_member_id: int = 200) -> list[LoanData]:
    """
    Reads raw transactional CSV data, validates integrity, and produces clean LoanData objects.
    Filters unreturned books (0), invalid member IDs, and corrupted epoch conversion dates.
    """
    cleaned_loans = []

    with open(filepath, 'r', encoding='utf-8-sig') as f:
        reader = csv.reader(f)
        for row in reader:
            if not row or len(row) < 4:
                continue
            try:
                b_id, m_id, d_loan, d_return = int(row[0]), int(row[1]), int(row[2]), int(row[3])
            except ValueError:
                continue  # Skip header or non-numeric rows

            # Validation Rules
            if d_return == 0:  # Book not returned
                continue
            if m_id > max_member_id:  # Invalid member ID noise
                continue

            loan = LoanData(b_id, m_id, d_loan, d_return)

            # Filter out corrupt conversion outputs
            if loan.date_returned_str in ('1899-12-30', '2024-01-01', 'Invalid Date'):
                continue

            cleaned_loans.append(loan)

    return cleaned_loans

## 4. Pipeline Execution & Analysis

In [4]:
# 1. Load Data
catalog = Bookshelf()
catalog.load_catalog(DATA_DIR / "books.csv")

cleaned_loans = process_loan_records(DATA_DIR / "bookloans.csv")

print(f"✅ Catalog Loaded: {len(catalog.books)} books.")
print(f"✅ Cleaned Loan Transactions: {len(cleaned_loans)} valid records.")

✅ Catalog Loaded: 120 books.
✅ Cleaned Loan Transactions: 2158 valid records.


## 5. Metric Aggregation & Reporting

In [5]:
# Metric Calculations
total_loans = len(cleaned_loans)
total_days_loaned = sum(l.days_loaned for l in cleaned_loans)
late_loans = [l for l in cleaned_loans if l.is_late]
total_late_days = sum(l.days_late for l in late_loans)

avg_loan_period = total_days_loaned / total_loans if total_loans else 0
late_percentage = (len(late_loans) / total_loans) * 100 if total_loans else 0
avg_late_period = total_late_days / len(late_loans) if late_loans else 0

# Genre & Popularity Tracking
genre_loan_counts = Counter()
book_duration_days = defaultdict(int)

for loan in cleaned_loans:
    book = catalog.get_book(loan.book_id)
    if book:
        genre_loan_counts[book.genre] += 1
        book_duration_days[book.title] += loan.days_loaned

# Output Report 1: Operational Summary
report_1_path = REPORTS_DIR / "library_report_2023.txt"
report_1_content = f"""==================================================
        LIBRARY OPERATIONAL SUMMARY REPORT (2023)
==================================================
Total Valid Transactions Processed : {total_loans}
Average Loan Duration               : {avg_loan_period:.2f} Days
Overdue Transaction Rate           : {late_percentage:.2f}%
Average Overdue Period (Late Items): {avg_late_period:.2f} Days
==================================================
"""

with open(report_1_path, 'w', encoding='utf-8') as f:
    f.write(report_1_content)

print(report_1_content)

# Output Report 2: Genre Preferences & Top Books
report_2_path = REPORTS_DIR / "reading_preferences_report.txt"
top_books = sorted(book_duration_days.items(), key=lambda x: x[1], reverse=True)[:5]

report_2_content = "==================================================\n"
report_2_content += "       GENRE PREFERENCES & POPULARITY REPORT\n"
report_2_content += "==================================================\n\n"
report_2_content += "LOANS BY GENRE:\n"
for genre, count in genre_loan_counts.most_common():
    report_2_content += f" - {genre:<15}: {count} loans\n"

report_2_content += "\nTOP 5 MOST BORROWED BOOKS (BY TOTAL DAYS):\n"
for rank, (title, days) in enumerate(top_books, start=1):
    report_2_content += f" {rank}. {title:<35} : {days} Total Days\n"
report_2_content += "==================================================\n"

with open(report_2_path, 'w', encoding='utf-8') as f:
    f.write(report_2_content)

print(report_2_content)

        LIBRARY OPERATIONAL SUMMARY REPORT (2023)
Total Valid Transactions Processed : 2158
Average Loan Duration               : 11.26 Days
Overdue Transaction Rate           : 33.46%
Average Overdue Period (Late Items): 3.97 Days

       GENRE PREFERENCES & POPULARITY REPORT

LOANS BY GENRE:
 - Fiction        : 677 loans
 - Non-fiction    : 550 loans
 - Tech           : 386 loans
 - Science        : 150 loans
 - Philosophy     : 100 loans
 - Biography      : 27 loans
 - Mystery        : 24 loans

TOP 5 MOST BORROWED BOOKS (BY TOTAL DAYS):
 1. The Hunger Games                    : 325 Total Days
 2. The Complete Mastermind             : 322 Total Days
 3. The Casual Vacancy                  : 316 Total Days
 4. Asami Asami                         : 304 Total Days
 5. Real Murders                        : 300 Total Days

